# Wakeword-Training Deutsch: Brav, Hier, Sitz, Such

Pipeline (Colab, T4):

1. Echte Aufnahmen einlesen, vereinheitlichen, Qualitätscheck (Validation-Sprecher bleibt draussen)
2. Deutsche TTS-Clips erzeugen (Piper, mehrere Stimmen)
3. TTS validieren: Whisper-Check, Ähnlichkeit zu den echten Clips, Stimmen-Statistik, manuelles Anhören der Grenzfälle
4. Trainingsordner bauen (echte + freigegebene TTS-Clips, Hard Negatives) und mit dem Original-`train.py` von openWakeWord trainieren
5. Auswerten am Validation-Sprecher, an den anderen Wörtern und an deutscher Sprache (Fehlaktivierungen pro Stunde)

Die wichtigsten Zwischenergebnisse (TTS-Clips, Whisper-Cache, deine Entscheidungen, fertige Modelle) liegen auf Google Drive, bereits Erledigtes wird übersprungen. Die Zellen lassen sich also nach einem Abbruch erneut ausführen. Die Konfiguration steht in Abschnitt 1.

Was bewusst anders ist als im Original-Notebook:

- Die TTS-Erzeugung von `train.py` (`--generate_clips`) wird nicht benutzt. Sie nutzt eine englische Stimme und englische Phonem-Gegenbeispiele. Stattdessen kommen die Clips aus diesem Notebook.
- `train.py` wird über einen kleinen Wrapper gestartet (`run_train.py`), der drei Kompatibilitätsprobleme umgeht (siehe Abschnitt 4).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0. Setup

In [ ]:
import subprocess, sys, os, shutil

def sh(cmd, check=True, cwd=None, env=None):
    """Shell-Befehl ausführen und Ausgabe live anzeigen."""
    print(f"$ {cmd}")
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"Befehl fehlgeschlagen (Exit {p.returncode}): {cmd}")
    return p.returncode

PIP = f"{sys.executable} -m pip -q install"

# datasets<4: neuere Versionen unterstützen Skript-Datasets (z. B. FMA) nicht mehr
sh(f"{PIP} 'datasets<4' soundfile transformers accelerate piper-tts ipywidgets pyyaml pandas")
# Abhängigkeiten, die openwakeword.data / train.py beim Import braucht
sh(f"{PIP} pronouncing mutagen torchinfo torchmetrics speechbrain audiomentations torch-audiomentations "
   f"onnx onnxscript onnxruntime scikit-learn scipy tqdm requests")

# openWakeWord aus dem Repo (liefert train.py); Abhängigkeiten oben selbst installiert
if not os.path.exists("/content/openwakeword"):
    sh("git clone --depth 1 https://github.com/dscripka/openwakeword /content/openwakeword")
sh(f"{PIP} --no-deps -e /content/openwakeword")
sh(f"{PIP} ai-edge-litert speexdsp-ns", check=False)  # optional, für das Training nicht nötig
sys.path.insert(0, "/content/openwakeword")

import openwakeword
from openwakeword.utils import download_models
download_models()  # Mel-Spektrogramm- und Embedding-Modell (werden für die Features gebraucht)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive nicht eingebunden (nicht in Colab?):", e)

$ /usr/bin/python3 -m pip -q install 'datasets<4' soundfile transformers accelerate piper-tts ipywidgets pyyaml pandas
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
$ /usr/bin/python3 -m pip -q install pronouncing mutagen torchinfo torchmetrics speechbrain audiomentations torch-audiomentations onnx onnxscript onnxruntime scikit-learn scipy tqdm requests
$ git clone --depth 1 https://github.com/dscripka/openwakeword /content/openwakeword
Cloning into '/content/openwakeword'...
$ /usr/bin/python3 -m pip -q install --no-deps -e /content/openwakeword
$ /usr/bin/python3 -m pip -q install ai-edge-litert speexdsp-ns
ERROR: Could not find a version that satisfies the requirement speexdsp-ns (from versions: none)
ERROR: No matching distribution found for speexdsp-ns


embedding_model.tflite: 100%|██████████| 1.33M/1.33M [00:00<00:00, 164MiB/s]
embedding_model.onnx: 100%|██████████| 1.33M/1.33M [00:00<00:00, 110MiB/s]
melspectrogram.tflite: 100%|██████████| 1.09M/1.09M [00:00<00:00, 158MiB/s]
melspectrogram.onnx: 100%|██████████| 1.09M/1.09M [00:00<00:00, 181MiB/s]
silero_vad.onnx: 100%|██████████| 1.81M/1.81M [00:00<00:00, 165MiB/s]
alexa_v0.1.tflite: 100%|██████████| 855k/855k [00:00<00:00, 169MiB/s]
alexa_v0.1.onnx: 100%|██████████| 854k/854k [00:00<00:00, 129MiB/s]
hey_mycroft_v0.1.tflite: 100%|██████████| 860k/860k [00:00<00:00, 128MiB/s]
hey_mycroft_v0.1.onnx: 100%|██████████| 858k/858k [00:00<00:00, 76.9MiB/s]
hey_jarvis_v0.1.tflite: 100%|██████████| 1.28M/1.28M [00:00<00:00, 139MiB/s]
hey_jarvis_v0.1.onnx: 100%|██████████| 1.27M/1.27M [00:00<00:00, 166MiB/s]
hey_rhasspy_v0.1.tflite: 100%|██████████| 416k/416k [00:00<00:00, 107MiB/s]
hey_rhasspy_v0.1.onnx: 100%|██████████| 204k/204k [00:00<00:00, 54.0MiB/s]
timer_v0.1.tflite: 100%|██████████| 

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Konfiguration

In [ ]:
import json, random, re, unicodedata, uuid, shutil
from math import gcd
from pathlib import Path
import numpy as np
import soundfile as sf
import pandas as pd
from scipy.signal import resample_poly
from tqdm.auto import tqdm
from IPython.display import display

# ---------------------------------------------------------------- Pfade
PROJECT_DIR = Path("/content/drive/MyDrive")   # bleibt erhalten (Drive)
DATA_ROOT   = PROJECT_DIR / "Aufnahmen"    # enthält Brav/ Hier/ Sitz/ Such/ (Validation-Ordner darin)
WORK        = Path("/content/work")        # lokale Platte (schnell, nach Sitzungsende weg)

# ---------------------------------------------------------------- Wörter
WORDS = ["Brav", "Hier", "Sitz", "Such"]
WORDS_TO_TRAIN = ["Brav"]       # zuerst ein Wort durchspielen, dann erweitern, z. B. ["Sitz", "Such", "Hier", "Brav"]
MODEL_SUFFIX = "_de"            # Modellname = wort + Suffix (z. B. sitz_de.onnx)
VALIDATION_DIR_NAMES = {"validation", "val", "validierung"}   # Ordnernamen (klein geschrieben) des Holdout-Sprechers
INCLUDE_VALIDATION_SPEAKER = True   # True nur für den finalen Lauf, wenn alle Einstellungen feststehen

# ---------------------------------------------------------------- Echte Aufnahmen
MIN_CLIP_S, MAX_CLIP_S = 0.15, 1.6   # Clips ausserhalb werden ausgeschlossen
PEAK_NORM_DBFS = -3.0                # Spitzenpegel aller Positiv-Clips (echt und TTS)
REAL_DUP = 10                        # Kopien jedes echten Trainingsclips (jede Kopie wird anders augmentiert)
REAL_TEST_FRAC = 0.2                 # Anteil der echten Trainingsclips, der für Early Stopping zurückgelegt wird
REAL_TEST_DUP = 5
REAL_NEG_DUP = 5                     # Kopien echter Clips der anderen Wörter (Negativbeispiele)
DROP_REAL_ASR_MISMATCH = False       # True: echte Clips ausschliessen, die Whisper nicht als Zielwort erkennt

# ---------------------------------------------------------------- TTS
N_TTS_POS = 1500          # TTS-Kandidaten pro trainiertem Wort (vor dem Filtern)
N_TTS_AS_NEG = 150        # TTS-Clips pro anderem Zielwort (dient als Negativ)
N_TTS_NEG = 50            # TTS-Clips pro Hard-Negative-Wort
VOICES = [  # (Piper-Stimme, Gewicht)
    ("de_DE-thorsten-high", 3), ("de_DE-thorsten-medium", 2), ("de_DE-thorsten_emotional-medium", 2),
    ("de_DE-mls-medium", 4), ("de_DE-kerstin-low", 1), ("de_DE-ramona-low", 1),
    ("de_DE-karlsson-low", 1), ("de_DE-pavoque-low", 1), ("de_DE-eva_k-x_low", 1),
]
EMOTION_ALLOW = {"neutral", "angry", "surprised", "amused"}   # nur für thorsten_emotional
LENGTH_SCALES = [0.8, 0.9, 1.0, 1.1, 1.25]
NEG_WORDS = [  # ähnlich klingende Wörter und andere Kommandos (Zielwörter werden automatisch entfernt)
    "Platz", "Fuß", "Komm", "Bleib", "Aus", "Nein", "Gut", "Stopp", "Warte", "Los", "Pfui", "Hallo", "Danke", "Nimm",
    "Spitz", "Blitz", "Witz", "Fritz", "Schatz", "Satz", "Sitzen", "Sitzung",
    "Buch", "Fluch", "Tuch", "Zug", "Besuch", "Suche", "Suchen", "Zucker", "Bruch",
    "Bier", "Tier", "Dir", "Wir", "Hierher", "Papier",
    "Bravo", "Graf", "Schaf", "Draht", "Brauch",
]

# ---------------------------------------------------------------- Validierung der TTS-Clips
ASR_MODEL = "openai/whisper-large-v3-turbo"   # transformers, GPU; schneller: "openai/whisper-small"
ASR_EXTRA_ACCEPT = {}                 # z. B. {"such": {"zuch"}}: zusätzlich akzeptierte Transkriptionen (klein, ohne Satzzeichen)
SIM_DROP_FRACTION = 0.10              # schwächste 10 % nach Ähnlichkeit zu den echten Clips werden automatisch verworfen
VOICE_MIN_PASS = 0.10                 # Stimmen mit geringerer Whisper-Trefferquote UND weniger als VOICE_REL_PASS x Median aller Stimmen werden gesperrt (ab 10 Clips)
VOICE_REL_PASS = 0.5
MIN_TTS_KEEP = 200                    # Mindestzahl TTS-Clips pro Zielwort: wird sie unterschritten, wird die Auswahl schrittweise gelockert
RESCUE_SIM_QUANTILE = 0.5             # Lockerungsstufe 2: Whisper-abgelehnte Clips behalten, deren Ähnlichkeit zu den echten Clips über diesem Quantil der bestandenen liegt
BLOCKED_VOICES = {}                   # manuell, z. B. {"sitz": ["de_DE-eva_k-x_low"]}
REVIEW_N_LOW_SIM, REVIEW_N_RESCUE, REVIEW_N_SPOT = 25, 15, 15
TTS_TEST_MAX = 300

# ---------------------------------------------------------------- Hintergrund und deutsche Sprache
EXTRA_BACKGROUND_DIRS = []            # eigene Geräusche (z. B. Roboter-Motor, 16 kHz WAV), werden zusätzlich untergemischt
FMA_HOURS = 1
GERMAN_SPEECH_DIR = PROJECT_DIR / "deutsche_sprache"   # optional: eigene Ordner mit deutscher Sprache (wav/flac)
GERMAN_SPEECH_HOURS = 4.0
GERMAN_HF_SOURCES = [  # (Dataset, Konfiguration, Split), werden der Reihe nach probiert
    ("facebook/voxpopuli", "de", "train"),
    ("google/fleurs", "de_de", "train"),
    ("facebook/multilingual_librispeech", "german", "train"),
]

# ---------------------------------------------------------------- Training
AUG_ROUNDS = 2
STEPS = 50000                 # fürs erste; für das finale Modell eher 50000
LAYER_SIZE = 32
MAX_NEGATIVE_WEIGHT = 1500
TARGET_FP_PER_HOUR = 0.2
BATCH_N = {"ACAV100M_sample": 1024, "german_speech": 256, "adversarial_negative": 50, "positive": 50}
SEED = 42
ACAV_SUBSET_ROWS = 800_000   # Zeilen der ACAV-Negativdaten (je 1,28 s, float16, ca. 2,5 GB = ca. 284 h). 0 = ganze Datei (2000 h, ca. 17 GB, sprengt den Colab-RAM)

# ---------------------------------------------------------------- abgeleitete Pfade
CACHE = PROJECT_DIR / "cache"
TTS_DIR = CACHE / "tts"
BG_DIR = CACHE / "background"
REAL_DIR = WORK / "real"
RUNS_DIR = WORK / "runs"
FEAT_DIR = WORK / "features"
STUBS = WORK / "stubs"
MODELS_OUT = PROJECT_DIR / "models"
for d in (CACHE, TTS_DIR, BG_DIR, REAL_DIR, RUNS_DIR, FEAT_DIR, STUBS, MODELS_OUT):
    d.mkdir(parents=True, exist_ok=True)

def slug(w):
    return (w.lower().replace("ß", "ss").replace("ä", "ae").replace("ö", "oe").replace("ü", "ue").replace(" ", "_"))

SR = 16000
AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}
TARGETS = {slug(w) for w in WORDS}
NEG_WORDS = [w for w in NEG_WORDS if slug(w) not in TARGETS]
ACCEPT = {slug(w): {slug(w)} | set(ASR_EXTRA_ACCEPT.get(slug(w), set())) for w in WORDS}
print("Trainiere:", WORDS_TO_TRAIN, "| Negativ-Wörter:", len(NEG_WORDS))

Trainiere: ['Brav'] | Negativ-Wörter: 42


## 2. Hilfsfunktionen (Audio)

In [ ]:
def to_mono_16k(data, sr):
    """Beliebiges Audio (1D oder (frames, kanäle), int oder float) nach float32, mono, 16 kHz."""
    data = np.asarray(data)
    if np.issubdtype(data.dtype, np.integer):
        data = data.astype(np.float32) / np.iinfo(data.dtype).max
    data = data.astype(np.float32)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if int(sr) != SR:
        g = gcd(int(sr), SR)
        data = resample_poly(data, SR // g, int(sr) // g).astype(np.float32)
    return data

def trim_silence(x, margin_start=0.12, margin_end=0.15, thresh_db=-40.0, frame=160):
    """Stille vorne/hinten auf eine kurze Marge kürzen (verlängert nie)."""
    n = len(x) // frame
    if n < 2:
        return x
    rms = np.sqrt(np.mean(x[:n * frame].reshape(n, frame) ** 2, axis=1) + 1e-12)
    db = 20 * np.log10(rms)
    active = np.flatnonzero(db > db.max() + thresh_db)
    if len(active) == 0:
        return x
    start = max(0, active[0] * frame - int(margin_start * SR))
    end = min(len(x), (active[-1] + 1) * frame + int(margin_end * SR))
    return x[start:end]

def peak_normalize(x, dbfs=PEAK_NORM_DBFS):
    peak = float(np.max(np.abs(x))) + 1e-9
    return (x * (10 ** (dbfs / 20) / peak)).astype(np.float32)

def to_int16(x):
    return np.clip(x * 32767.0, -32768, 32767).astype(np.int16)

def save_wav(path, x):
    sf.write(str(path), to_int16(x), SR, subtype="PCM_16")

def load_wav16k(path):
    x, sr = sf.read(str(path), dtype="float32", always_2d=False)
    return to_mono_16k(x, sr)

def clip_stats(x):
    peak = float(np.max(np.abs(x))) + 1e-9
    return dict(dur=len(x) / SR, peak_db=20 * np.log10(peak),
                rms_db=20 * np.log10(float(np.sqrt(np.mean(x ** 2))) + 1e-9),
                clipped=float(np.mean(np.abs(x) >= 0.999)))

def norm_text(t):
    return re.sub(r"[^a-zäöüß]", "", unicodedata.normalize("NFC", t).lower())

SUBSET_SCRIPT = """
import sys, os, numpy as np
from numpy.lib.format import open_memmap
full, out, rows, seed = sys.argv[1], sys.argv[2], int(sys.argv[3]), int(sys.argv[4])
src = np.load(full, mmap_mode="r")
CH = 64
n_chunks = src.shape[0] // CH
want = min(n_chunks, rows // CH)
rng = np.random.default_rng(seed)
pick = rng.choice(n_chunks, size=want, replace=False)   # zufällige Blöcke aus der ganzen Datei, in zufälliger Reihenfolge
dst = open_memmap(out + ".tmp.npy", mode="w+", dtype=src.dtype, shape=(want * CH,) + src.shape[1:])
for i, c in enumerate(pick):
    dst[i * CH:(i + 1) * CH] = src[c * CH:(c + 1) * CH]
    if i % 2000 == 0:
        dst.flush()
        print(i, "/", want, flush=True)
dst.flush()
del dst
os.replace(out + ".tmp.npy", out)
print("fertig:", want * CH, "Zeilen", flush=True)
"""

def prepare_acav_subset():
    """Kleine, gemischte Stichprobe der ACAV-Negativdaten (feste Grösse im RAM statt 17 GB, die beim Training nach und nach gelesen werden)."""
    full = FEAT_DIR / "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
    if not ACAV_SUBSET_ROWS:
        return full
    sub = FEAT_DIR / f"acav_subset_{ACAV_SUBSET_ROWS}.npy"
    if sub.exists():
        return sub
    assert full.exists(), "ACAV-Datei fehlt: Abschnitt 3 ausführen"
    script = STUBS / "make_subset.py"
    script.write_text(SUBSET_SCRIPT)
    sh(f"{sys.executable} {script} {full} {sub} {ACAV_SUBSET_ROWS} {SEED}")   # eigener Prozess: Speicher wird danach wieder frei
    return sub

## 3. Hintergrundgeräusche, Impulsantworten und vorberechnete Features
Wie im Original-Notebook (AudioSet, FMA, MIT-Impulsantworten, ACAV100M-Features). Wird auf Drive bzw. lokal zwischengespeichert. Der ACAV-Download ist gross und liegt nur lokal, er wird in jeder neuen Sitzung neu geladen.

In [ ]:
import datasets

def prepare_rirs():
    out = BG_DIR / "mit_rirs"
    if out.exists() and any(out.iterdir()):
        return out
    out.mkdir(parents=True, exist_ok=True)
    ds = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    for row in tqdm(ds, desc="MIT RIRs"):
        x = to_mono_16k(row["audio"]["array"], row["audio"]["sampling_rate"])
        save_wav(out / Path(row["audio"]["path"]).name, x)
    return out

AUDIOSET_CLIPS = 1000   # ca. 10 s pro Clip, also knapp 3 Stunden Hintergrundgeräusche

def prepare_audioset(n_clips=AUDIOSET_CLIPS):
    # Das Repo agkphysics/AudioSet liegt inzwischen als Parquet vor (die alten .tar-Dateien gibt es nicht mehr, daher 404).
    out = BG_DIR / "audioset_16k"
    if out.exists() and any(out.iterdir()):
        return out
    out.mkdir(parents=True, exist_ok=True)
    ds = datasets.load_dataset("agkphysics/AudioSet", "balanced", split="train", streaming=True)
    ds = ds.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for i, row in enumerate(tqdm(ds, total=n_clips, desc="AudioSet")):
        if i >= n_clips:
            break
        x = to_mono_16k(row["audio"]["array"], 16000)
        save_wav(out / f"audioset_{i:05d}.wav", x)
    return out

def prepare_fma(hours=FMA_HOURS):
    out = BG_DIR / "fma"
    if out.exists() and any(out.iterdir()):
        return out
    out.mkdir(parents=True, exist_ok=True)
    try:
        ds = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True, trust_remote_code=True)
        ds = ds.cast_column("audio", datasets.Audio(sampling_rate=16000))
        for i, row in enumerate(tqdm(ds, total=int(hours * 3600 // 30), desc="FMA")):
            if i >= int(hours * 3600 // 30):
                break
            save_wav(out / (Path(row["audio"]["path"]).stem + ".wav"), to_mono_16k(row["audio"]["array"], 16000))
    except Exception as e:
        print("FMA übersprungen:", repr(e)[:200])
    return out

RIR_DIR = prepare_rirs()
AUDIOSET_DIR = prepare_audioset()
FMA_DIR = prepare_fma()
BACKGROUND_DIRS = [str(d) for d in [AUDIOSET_DIR, FMA_DIR] if any(Path(d).iterdir())] + [str(d) for d in EXTRA_BACKGROUND_DIRS]
print("Hintergrundordner:", BACKGROUND_DIRS)

HF_TOKEN = os.environ.get("HF_TOKEN", "")   # optional; Token nie in den Code schreiben
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""   # Colab-Secret "HF_TOKEN" (Notebook-Zugriff aktivieren)
    except Exception:
        pass
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
AUTH = f"--header='Authorization: Bearer {HF_TOKEN}' " if HF_TOKEN else ""

ACAV = FEAT_DIR / "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
FPVAL = FEAT_DIR / "validation_set_features.npy"
base = "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/"
if not FPVAL.exists():
    sh(f"wget {AUTH}-c -nv -O {FPVAL} {base}validation_set_features.npy")
if not ACAV.exists():
    sh(f"wget {AUTH}-c -nv -O {ACAV} {base}openwakeword_features_ACAV100M_2000_hrs_16bit.npy")

README.md:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

fma.py:   0%|          | 0.00/46.3k [00:00<?, ?B/s]

FMA übersprungen: ValueError('Cannot seek streaming HTTP file')
Hintergrundordner: ['/content/drive/MyDrive/cache/background/audioset_16k']
$ wget -c -nv -O /content/work/features/validation_set_features.npy https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
2026-11-18 17:12:44 URL:https://us.gcp.cdn.hf.co/xet-bridge-us/64f3a0b6918ffcc15af6923c/d06b67aa405c24aab66854c32fe850f58387e668e24d58f9c1885d81d86c94cd?xip=K-aQmGwzZDA&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27validation_set_features.npy%3B+filename%3D%22validation_set_features.npy%22%3B&X-Xet-Cas-Uid=public&user_id=public&Expires=1795025562&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRmM2EwYjY5MThmZmNjMTVhZjY5MjNjL2QwNmI2N2FhNDA1YzI0YWFiNjY4NTRjMzJmZTg1MGY1ODM4N2U2NjhlMjRkNThmOWMxODg1ZDgxZDg2Yzk0Y2RcXD94aXA9Sy1hUW1Hd3paREEmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qJlgtWGV0LUNhcy1VaWQ9cHVibGljJnVzZXJfaWQ9

## 4. Wrapper für `train.py`
`train.py` importiert beim Start unbedingt `generate_samples` (Piper) und `openwakeword.data` importiert `acoustics`. Beides brauchen wir nicht (und `acoustics` läuft mit aktuellem SciPy nicht). Der Wrapper legt deshalb leere Platzhalter an und behebt zwei weitere Probleme neuer PyTorch-Versionen: `torchaudio.load` benötigt teils `torchcodec`, und `torch.onnx.export` nutzt standardmässig den neuen Exporter. Die Trainingslogik von `train.py` bleibt unverändert.

In [ ]:
(STUBS / "acoustics").mkdir(exist_ok=True)
(STUBS / "acoustics" / "__init__.py").write_text('"""Platzhalter: wird nur von mix_clips_batch gebraucht, das hier nicht benutzt wird."""\n')
(STUBS / "generate_samples.py").write_text(
    "def generate_samples(*args, **kwargs):\n"
    "    raise RuntimeError('TTS-Erzeugung von train.py ist deaktiviert: Clips werden vorab im Notebook erzeugt.')\n")
(STUBS / "run_train.py").write_text(r'''
import os, sys, inspect
HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
try:
    import numpy as np, soundfile as sf, torch, torchaudio

    class _Info:
        def __init__(self, i):
            self.sample_rate, self.num_frames, self.num_channels = i.samplerate, i.frames, i.channels
            self.bits_per_sample, self.encoding = 16, "PCM_S"

    def _info(path, *a, **k):
        return _Info(sf.info(str(path)))

    def _load(path, frame_offset=0, num_frames=-1, normalize=True, channels_first=True, *a, **k):
        data, sr = sf.read(str(path), start=int(frame_offset), frames=int(num_frames), dtype="float32", always_2d=True)
        return torch.from_numpy(np.ascontiguousarray(data.T if channels_first else data)), sr

    torchaudio.load, torchaudio.info = _load, _info
except Exception as e:
    print("torchaudio-Patch übersprungen:", e)
try:
    import torch
    _orig = torch.onnx.export
    if "dynamo" in inspect.signature(_orig).parameters:
        def _export(*a, **k):
            k.setdefault("dynamo", False)
            return _orig(*a, **k)
        torch.onnx.export = _export
except Exception as e:
    print("onnx-Patch übersprungen:", e)

# Speicheranzeige: alle 60 Sekunden eine Zeile mit freiem RAM, Speicher des Trainingsprozesses und seiner Worker
import threading, time
def _monitor():
    try:
        import psutil
    except Exception:
        return
    me = psutil.Process()
    while True:
        try:
            vm = psutil.virtual_memory()
            kids = sum(c.memory_info().rss for c in me.children(recursive=True))
            print()
            print(f"[RAM] frei {vm.available / 1e9:.1f} GB | Training {me.memory_info().rss / 1e9:.1f} GB | Worker {kids / 1e9:.1f} GB", flush=True)
        except Exception:
            pass
        time.sleep(60)
threading.Thread(target=_monitor, daemon=True).start()

# Speicher-Patch: train.py baut aus den 11 h Validierungsdaten einen ca. 3 GB grossen Array und kopiert ihn bei jeder
# Validierung nochmals zu einem Batch. Das sprengt den Colab-RAM (Exit 137 bei der ersten Validierung).
# Stattdessen werden die Fenster blockweise erzeugt. Alle Verbraucher in train.py summieren die Treffer über die Blöcke.
FP_CODE = """X_val_fp_feats = np.load(config["false_positive_validation_data_path"], mmap_mode="r")
        class _FPWindows:
            # erzeugt die Validierungsfenster blockweise statt als ein 3-GB-Array
            def __init__(self, feats, n, bs=10000):
                self.f, self.n, self.bs = feats, n, bs
            def __iter__(self):
                total = self.f.shape[0] - self.n
                for s in range(0, total, self.bs):
                    e = min(total, s + self.bs)
                    seg = np.asarray(self.f[s:e + self.n - 1], dtype=np.float32)
                    win = np.lib.stride_tricks.sliding_window_view(seg, (self.n, seg.shape[1]))[:, 0]
                    yield torch.from_numpy(np.ascontiguousarray(win)), torch.zeros(e - s, dtype=torch.float32)
        X_val_fp = _FPWindows(X_val_fp_feats, input_shape[0])

"""
script = "/content/openwakeword/openwakeword/train.py"
src = open(script, encoding="utf-8").read()
start = src.index('X_val_fp = np.load(config["false_positive_validation_data_path"])')
end = src.index("        X_val_pos = np.load(")
assert 0 < start < end, "train.py hat sich geändert: Speicher-Patch nicht anwendbar"
src = src[:start] + FP_CODE + src[end:]
# train.py hat default="False" (Text, gilt als wahr): die tflite-Umwandlung lief deshalb immer und scheitert ohne onnx_tf.
# Das .onnx-Modell ist zu diesem Zeitpunkt schon gespeichert; tflite wird nicht gebraucht.
conv = "        if args.convert_to_tflite:"
assert src.count(conv) == 1, "train.py hat sich geändert: tflite-Patch nicht anwendbar"
src = src.replace(conv, "        if False:  # tflite-Umwandlung deaktiviert")
sys.argv[0] = script
exec(compile(src, script, "exec"), {"__name__": "__main__", "__file__": script})
''')
print("Wrapper geschrieben:", STUBS / "run_train.py")

Wrapper geschrieben: /content/work/stubs/run_train.py


## 5. Echte Aufnahmen einlesen und prüfen
Die Dateien der Wortordner werden rekursiv gelesen. Alles unter einem Ordner `Validation` ist der Holdout-Sprecher und wird nie zum Training verwendet. Clips werden auf 16 kHz mono gebracht, Stille wird auf eine kurze Marge gekürzt, der Pegel vereinheitlicht.

In [ ]:
def find_word_dir(root, word):
    for p in Path(root).iterdir():
        if p.is_dir() and p.name.lower() == word.lower():
            return p
    raise FileNotFoundError(f"Kein Ordner '{word}' in {root}")

def split_train_val(word_dir):
    train, val = [], []
    for p in sorted(Path(word_dir).rglob("*")):
        if p.suffix.lower() not in AUDIO_EXTS or not p.is_file():
            continue
        parts = {q.lower() for q in p.relative_to(word_dir).parts[:-1]}
        (val if parts & VALIDATION_DIR_NAMES else train).append(p)
    return train, val

def import_real_clips():
    rows = []
    for w in WORDS:
        train, val = split_train_val(find_word_dir(DATA_ROOT, w))
        for split, files in (("train", train), ("val", val)):
            out = REAL_DIR / slug(w) / split
            if out.exists():
                shutil.rmtree(out)
            out.mkdir(parents=True)
            for i, p in enumerate(files):
                data, sr = sf.read(str(p), dtype="float32", always_2d=True)
                x = trim_silence(to_mono_16k(data, sr))
                st = clip_stats(x)
                flags = []
                if st["dur"] < MIN_CLIP_S: flags.append("zu kurz")
                if st["dur"] > MAX_CLIP_S: flags.append("zu lang")
                if st["peak_db"] < -50: flags.append("stumm")
                elif st["peak_db"] < -35: flags.append("sehr leise")
                if st["clipped"] > 0.01: flags.append("übersteuert")
                dst = None
                if not ({"zu kurz", "zu lang", "stumm"} & set(flags)):
                    dst = out / f"{split}_{i:03d}_{re.sub(r'[^A-Za-z0-9]+', '_', p.stem)[:40]}.wav"
                    save_wav(dst, peak_normalize(x))
                rows.append(dict(word=w, split=split, src=str(p), dst=str(dst) if dst else "", flags=", ".join(flags), **st))
    return pd.DataFrame(rows)

df_real = import_real_clips()
print(df_real.groupby(["word", "split"]).agg(n=("src", "count"), kept=("dst", lambda s: (s != "").sum()),
                                               dur_mean=("dur", "mean")).round(2))
bad = df_real[df_real["flags"] != ""]
if len(bad):
    print("\nAuffällige Clips:")
    display(bad[["word", "split", "src", "flags", "dur", "peak_db"]].round(2))

             n  kept  dur_mean
word split                    
Brav train  75    75      0.60
     val    11    11      0.51
Hier train  78    78      0.49
     val    12    12      0.47
Sitz train  82    81      0.76
     val    10    10      0.65
Such train  78    78      0.74
     val    10    10      0.57

Auffällige Clips:


,word,split,src,flags,dur,peak_db
225,Sitz,train,/content/drive/MyDrive/Aufnahmen/Sitz/Sitz_1.wav,zu lang,2.03,-9.66


## 6. TTS-Clips erzeugen (Piper, deutsche Stimmen)
Pro Clip werden Stimme, Sprecher (bei Multi-Speaker-Stimmen), Sprechtempo und Zufallsparameter variiert. Die Erzeugung ist fortsetzbar: vorhandene Clips zählen mit. Die Dateinamen enthalten Stimme und Sprecher, damit die Statistik pro Stimme möglich ist.

In [ ]:
from piper import PiperVoice, SynthesisConfig

def load_voices():
    voice_dir = CACHE / "voices"
    voice_dir.mkdir(exist_ok=True)
    out = {}
    for name, weight in VOICES:
        onnx = voice_dir / f"{name}.onnx"
        if not onnx.exists():
            sh(f"{sys.executable} -m piper.download_voices {name} --download-dir {voice_dir}", check=False)
        if not onnx.exists():
            print("Stimme übersprungen (Download fehlgeschlagen):", name)
            continue
        try:
            v = PiperVoice.load(str(onnx), use_cuda=False)
        except Exception as e:
            print("Stimme übersprungen:", name, repr(e)[:150])
            continue
        n_spk = v.config.num_speakers
        speakers = list(range(n_spk)) if n_spk > 1 else [None]
        if "emotional" in name and v.config.speaker_id_map:
            ids = [i for n, i in v.config.speaker_id_map.items() if n.lower() in EMOTION_ALLOW]
            speakers = ids or speakers
        out[name] = dict(voice=v, weight=weight, speakers=speakers)
    print("Geladene Stimmen:", {k: len(v["speakers"]) for k, v in out.items()})
    return out

def synth(v, text, speaker_id, length_scale, noise_scale, noise_w):
    cfg = SynthesisConfig(speaker_id=speaker_id, length_scale=length_scale, noise_scale=noise_scale,
                          noise_w_scale=noise_w, normalize_audio=True)
    chunks = [np.asarray(c.audio_float_array).reshape(-1) for c in v.synthesize(text, syn_config=cfg)]
    if not chunks:
        return np.zeros(0, np.float32)
    return to_mono_16k(np.concatenate(chunks), v.config.sample_rate)

def generate_tts(voices, word, n_target):
    out = TTS_DIR / slug(word)
    out.mkdir(parents=True, exist_ok=True)
    existing = len(list(out.glob("*.wav")))
    if existing >= n_target:
        return
    rng = random.Random(f"{word}-{existing}")
    names = list(voices)
    weights = [voices[n]["weight"] for n in names]
    made, attempts = existing, 0
    pbar = tqdm(total=n_target - existing, desc=f"TTS {word}", leave=False)
    while made < n_target and attempts < 4 * (n_target - existing):
        attempts += 1
        name = rng.choices(names, weights)[0]
        spk = rng.choice(voices[name]["speakers"])
        ls = rng.choice(LENGTH_SCALES)
        text = word + rng.choice(["!", ".", "", "!"])
        try:
            x = synth(voices[name]["voice"], text, spk, ls, rng.uniform(0.4, 1.0), rng.uniform(0.4, 1.0))
        except Exception as e:
            print("Synthese-Fehler:", name, repr(e)[:120])
            continue
        if len(x) < 0.1 * SR:
            continue
        x = peak_normalize(trim_silence(x))
        save_wav(out / f"{name}__spk{spk if spk is not None else 0}__ls{ls:.2f}__{uuid.uuid4().hex[:8]}.wav", x)
        made += 1
        pbar.update(1)
    pbar.close()

VOICE_POOL = load_voices()
if not VOICE_POOL:
    raise RuntimeError('Keine Piper-Stimme geladen (Download blockiert?). Netzwerk und VOICES prüfen.')
tts_plan = {}
for w in WORDS:
    tts_plan[w] = N_TTS_POS if w in WORDS_TO_TRAIN else N_TTS_AS_NEG
for w in NEG_WORDS:
    tts_plan[w] = N_TTS_NEG
for w, n in tts_plan.items():
    generate_tts(VOICE_POOL, w, n)
print({w: len(list((TTS_DIR / slug(w)).glob("*.wav"))) for w in tts_plan})

Geladene Stimmen: {'de_DE-thorsten-high': 1, 'de_DE-thorsten-medium': 1, 'de_DE-thorsten_emotional-medium': 4, 'de_DE-mls-medium': 236, 'de_DE-kerstin-low': 1, 'de_DE-ramona-low': 1, 'de_DE-karlsson-low': 1, 'de_DE-pavoque-low': 1, 'de_DE-eva_k-x_low': 1}
{'Brav': 1500, 'Hier': 1500, 'Sitz': 1500, 'Such': 3000, 'Platz': 50, 'Fuß': 50, 'Komm': 50, 'Bleib': 50, 'Aus': 50, 'Nein': 50, 'Gut': 50, 'Stopp': 50, 'Warte': 50, 'Los': 50, 'Pfui': 50, 'Hallo': 50, 'Danke': 50, 'Nimm': 50, 'Spitz': 50, 'Blitz': 50, 'Witz': 50, 'Fritz': 50, 'Schatz': 50, 'Satz': 50, 'Sitzen': 50, 'Sitzung': 50, 'Buch': 50, 'Fluch': 50, 'Tuch': 50, 'Zug': 50, 'Besuch': 50, 'Suche': 50, 'Suchen': 50, 'Zucker': 50, 'Bruch': 50, 'Bier': 50, 'Tier': 50, 'Dir': 50, 'Wir': 50, 'Hierher': 50, 'Papier': 50, 'Bravo': 50, 'Graf': 50, 'Schaf': 50, 'Draht': 50, 'Brauch': 50}


## 7. Validierung 1: Whisper-Check
Jeder Clip (echt und TTS) wird mit faster-whisper auf Deutsch transkribiert. Ein Clip gilt als verständlich, wenn die normalisierte Transkription dem Wort entspricht. Das prüft Verständlichkeit, nicht Natürlichkeit, und kann auch gute Clips fälschlich ablehnen. Für Ein-Wort-Clips ist Whisper nicht perfekt, deshalb gibt es `ASR_EXTRA_ACCEPT` und das manuelle Anhören danach.

In [ ]:
import torch
from transformers import pipeline

def load_asr():
    """Whisper auf der GPU über transformers (nutzt das vorhandene PyTorch/CUDA, kein cuDNN-Ärger wie bei ctranslate2)."""
    if not torch.cuda.is_available():
        print("WARNUNG: keine GPU aktiv (Laufzeit > Laufzeittyp ändern > T4). Auf der CPU wäre das extrem langsam; nehme 'openai/whisper-small'.")
        return pipeline("automatic-speech-recognition", model="openai/whisper-small", device=-1)
    asr = pipeline("automatic-speech-recognition", model=ASR_MODEL, torch_dtype=torch.float16, device="cuda:0")
    print("Whisper:", ASR_MODEL, "auf", torch.cuda.get_device_name(0))
    return asr

ASR_BATCH = 32
# Alte Konfigurationswerte (faster-whisper-Namen wie "large-v3-turbo") auf Hugging-Face-Namen abbilden
if "/" not in ASR_MODEL:
    ASR_MODEL = "openai/whisper-" + ASR_MODEL
ASR = load_asr()
ASR_CACHE_FILE = CACHE / "asr_cache.json"
ASR_CACHE = json.loads(ASR_CACHE_FILE.read_text()) if ASR_CACHE_FILE.exists() else {}

def transcribe_batch(xs):
    pad = np.zeros(int(0.5 * SR), np.float32)
    inputs = [{"raw": np.concatenate([pad, x.astype(np.float32), pad]), "sampling_rate": SR} for x in xs]
    outs = ASR(inputs, batch_size=ASR_BATCH,
               generate_kwargs={"language": "german", "task": "transcribe", "num_beams": 1, "do_sample": False})
    return [dict(text=o["text"].strip(), norm=norm_text(o["text"]), lp=0.0) for o in outs]

def asr_key(p):
    p = Path(p)
    for base in (TTS_DIR, REAL_DIR):
        try:
            return f"{base.name}/{p.relative_to(base)}@{p.stat().st_size}"
        except ValueError:
            pass
    return str(p)

def run_asr(paths, chunk=256):
    todo = [p for p in paths if asr_key(p) not in ASR_CACHE]
    for i in tqdm(range(0, len(todo), chunk), desc="Whisper (Blöcke à %d)" % chunk):
        part = todo[i:i + chunk]
        for p, r in zip(part, transcribe_batch([load_wav16k(p) for p in part])):
            ASR_CACHE[asr_key(p)] = r
        ASR_CACHE_FILE.write_text(json.dumps(ASR_CACHE, ensure_ascii=False))   # nach jedem Block gesichert
    return {str(p): ASR_CACHE[asr_key(p)] for p in paths}

# --- echte Clips
real_paths = [Path(p) for p in df_real["dst"] if p]
asr_real = run_asr(real_paths)
df_real["asr"] = df_real["dst"].map(lambda p: asr_real[p]["text"] if p else "")
df_real["asr_ok"] = [bool(d) and asr_real[d]["norm"] in ACCEPT[slug(w)] for d, w in zip(df_real["dst"], df_real["word"])]
print("Whisper-Trefferquote echte Clips:")
print(df_real[df_real["dst"] != ""].groupby(["word", "split"])["asr_ok"].mean().round(2))
mism = df_real[(df_real["dst"] != "") & (~df_real["asr_ok"])]
if len(mism):
    print("\nEchte Clips, die Whisper nicht als Zielwort erkennt (zum Anhören; sie bleiben im Training, ausser DROP_REAL_ASR_MISMATCH):")
    display(mism[["word", "split", "src", "asr"]])
DROPPED_REAL = set(Path(p) for p in mism["dst"]) if DROP_REAL_ASR_MISMATCH else set()

# --- TTS-Clips
rows = []
for w in tts_plan:
    for p in sorted((TTS_DIR / slug(w)).glob("*.wav")):
        voice, spk = p.name.split("__")[:2]
        rows.append(dict(word=w, slug=slug(w), path=str(p), voice=voice, spk=spk))
df_tts = pd.DataFrame(rows)
asr_tts = run_asr([Path(p) for p in df_tts["path"]])
df_tts["asr"] = df_tts["path"].map(lambda p: asr_tts[p]["text"])
df_tts["norm"] = df_tts["path"].map(lambda p: asr_tts[p]["norm"])
df_tts["is_target"] = df_tts["slug"].isin(TARGETS)
df_tts["asr_ok"] = [n in ACCEPT[s] if s in ACCEPT else False for n, s in zip(df_tts["norm"], df_tts["slug"])]

tgt = df_tts[df_tts["is_target"]]
print("\nWhisper-Trefferquote TTS (Zielwörter) pro Wort:")
print(tgt.groupby("word")["asr_ok"].agg(["mean", "count"]).round(2))
print("\nTrefferquote pro Wort und Stimme:")
print(tgt.pivot_table(index="voice", columns="word", values="asr_ok", aggfunc="mean").round(2))

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.62GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

... (11 outputs left out)

,word,split,src,asr
0,Brav,train,/content/drive/MyDrive/Aufnahmen/Brav/Brav 01.wav,Und auf.
1,Brav,train,/content/drive/MyDrive/Aufnahmen/Brav/Brav 02.wav,Bravo!
4,Brav,train,/content/drive/MyDrive/Aufnahmen/Brav/Brav 05.wav,Bravo!
5,Brav,train,/content/drive/MyDrive/Aufnahmen/Brav/Brav 06.wav,Braaf!
7,Brav,train,/content/drive/MyDrive/Aufnahmen/Brav/Brav 08.wav,Bravo!
...,...,...,...,...
348,Such,val,/content/drive/MyDrive/Aufnahmen/Such/Validati...,Sog.
352,Such,val,/content/drive/MyDrive/Aufnahmen/Such/Validati...,Soch.
353,Such,val,/content/drive/MyDrive/Aufnahmen/Such/Validati...,Zug.
354,Such,val,/content/drive/MyDrive/Aufnahmen/Such/Validati...,Zug.


Whisper (Blöcke à 256):   0%|          | 0/3 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Whisper-Trefferquote TTS (Zielwörter) pro Wort:
      mean  count
word             
Brav  0.28   1500
Hier  0.26   1500
Sitz  0.30   1500
Such  0.08   3000

Trefferquote pro Wort und Stimme:
word                             Brav  Hier  Sitz  Such
voice                                                  
de_DE-eva_k-x_low                0.04  0.00  0.00  0.02
de_DE-karlsson-low               0.01  0.26  0.42  0.00
de_DE-kerstin-low                0.00  0.00  0.00  0.00
de_DE-mls-medium                 0.00  0.00  0.00  0.00
de_DE-pavoque-low                0.62  0.02  0.01  0.00
de_DE-ramona-low                 0.00  0.00  0.01  0.00
de_DE-thorsten-high              0.61  0.78  0.63  0.24
de_DE-thorsten-medium            0.96  0.63  0.90  0.21
de_DE-thorsten_emotional-medium  0.02  0.10  0.42  0.03


## 8. Validierung 2: Ähnlichkeit zu den echten Clips
Für jeden Clip wird das openWakeWord-Embedding berechnet (2-Sekunden-Fenster, Wort am Ende ausgerichtet, wie im Training). Die Ähnlichkeit (Cosinus) misst den Abstand zum Mittelwert der echten Trainingsclips desselben Wortes. Die schwächsten `SIM_DROP_FRACTION` der Whisper-bestandenen Clips fallen automatisch raus. Zusätzlich werden Stimmen mit sehr niedriger Whisper-Trefferquote pro Wort gesperrt.

In [ ]:
from openwakeword.utils import AudioFeatures
F = AudioFeatures(device="cpu")
WIN, END_PAD = 32000, 1600

def embed_flat(paths, batch=64):
    out = []
    for i in tqdm(range(0, len(paths), batch), desc="Embeddings", leave=False):
        buf = np.zeros((len(paths[i:i + batch]), WIN), np.int16)
        for j, p in enumerate(paths[i:i + batch]):
            x = to_int16(load_wav16k(p))[: WIN - END_PAD]
            buf[j, WIN - END_PAD - len(x): WIN - END_PAD] = x
        e = F.embed_clips(buf, batch_size=64, ncpu=2)
        out.append(e.reshape(len(buf), -1))
    e = np.vstack(out)
    return e / (np.linalg.norm(e, axis=1, keepdims=True) + 1e-9)

df_tts["sim"] = np.nan
real_sim_report = {}
for w in WORDS:
    real_files = sorted((REAL_DIR / slug(w) / "train").glob("*.wav"))
    real_files = [p for p in real_files if p not in DROPPED_REAL]
    if len(real_files) < 3:
        print(f"{w}: zu wenige echte Clips für die Ähnlichkeit"); continue
    er = embed_flat(real_files)
    csum = er.sum(axis=0)
    loo = [(er[i] @ (csum - er[i])) / (np.linalg.norm(csum - er[i]) + 1e-9) for i in range(len(er))]
    cen = csum / np.linalg.norm(csum)
    real_sim_report[w] = (np.mean(loo), np.percentile(loo, 5))
    m = df_tts["word"] == w
    if m.any():
        et = embed_flat([Path(p) for p in df_tts.loc[m, "path"]])
        df_tts.loc[m, "sim"] = et @ cen
print("Echte Clips untereinander (Leave-one-out): Mittel / 5. Perzentil:",
      {w: tuple(round(float(v), 2) for v in r) for w, r in real_sim_report.items()})

# --- automatische Entscheidung für Zielwort-TTS (weich: Stimmen werden nur relativ zu den anderen gesperrt, mit Lockerung bei zu wenigen Clips)
df_tts["voice_blocked"] = False
for w in WORDS:
    s = slug(w)
    g = df_tts[(df_tts["slug"] == s)].groupby("voice")["asr_ok"].agg(["mean", "count"])
    med = g["mean"].median() if len(g) else 0
    print(f"{w}: Whisper-Quote je Stimme:", {v: round(float(r['mean']), 2) for v, r in g.iterrows()})
    for voice, r in g.iterrows():
        weak = r["count"] >= 10 and r["mean"] < VOICE_MIN_PASS and r["mean"] < VOICE_REL_PASS * med
        if weak or voice in BLOCKED_VOICES.get(s, []):
            df_tts.loc[(df_tts["slug"] == s) & (df_tts["voice"] == voice), "voice_blocked"] = True
            print(f"  Stimme gesperrt für {w}: {voice} (Whisper-Quote {r['mean']:.2f}, Median {med:.2f})")

df_tts["low_sim"] = False
df_tts["rescued"] = False
for w in WORDS:
    s = slug(w)
    m = (df_tts["word"] == w) & df_tts["asr_ok"] & ~df_tts["voice_blocked"] & df_tts["sim"].notna()
    if m.sum() >= 20:
        thr = df_tts.loc[m, "sim"].quantile(SIM_DROP_FRACTION)
        df_tts.loc[m & (df_tts["sim"] < thr), "low_sim"] = True
    cur = lambda: int((df_tts["is_target"] & (df_tts["word"] == w) & df_tts["asr_ok"] & ~df_tts["voice_blocked"] & ~df_tts["low_sim"]).sum()) \
        + int((df_tts["word"] == w).__and__(df_tts["rescued"]).sum())
    if df_tts["word"].eq(w).any() and df_tts.loc[df_tts["word"] == w, "is_target"].iloc[0] and cur() < MIN_TTS_KEEP:
        # Stufe 1: nur manuell gesperrte Stimmen bleiben gesperrt, automatisch gesperrte werden freigegeben, schwache Ähnlichkeit wird nicht mehr verworfen
        auto_blocked = df_tts["voice_blocked"] & ~df_tts["voice"].isin(BLOCKED_VOICES.get(s, []))
        df_tts.loc[(df_tts["word"] == w) & auto_blocked, "voice_blocked"] = False
        df_tts.loc[df_tts["word"] == w, "low_sim"] = False
        print(f"  {w}: nur {cur()} Clips, Stimmensperre und Ähnlichkeitsfilter aufgehoben.")
    if df_tts.loc[df_tts["word"] == w, "is_target"].any() and cur() < MIN_TTS_KEEP:
        # Stufe 2: von Whisper abgelehnte Clips behalten, die den echten Aufnahmen akustisch ähnlich sind
        ok_sim = df_tts.loc[(df_tts["word"] == w) & df_tts["asr_ok"] & df_tts["sim"].notna(), "sim"]
        if len(ok_sim) >= 5:
            thr = ok_sim.quantile(RESCUE_SIM_QUANTILE)
        else:
            thr = df_tts.loc[(df_tts["word"] == w) & df_tts["sim"].notna(), "sim"].quantile(0.5)
        cand = (df_tts["word"] == w) & ~df_tts["asr_ok"] & ~df_tts["voice"].isin(BLOCKED_VOICES.get(s, [])) & (df_tts["sim"] >= thr)
        df_tts.loc[cand, "rescued"] = True
        print(f"  {w}: {int(cand.sum())} von Whisper abgelehnte, aber ähnliche Clips zusätzlich behalten (bitte beim Anhören prüfen).")
        top = df_tts[(df_tts["word"] == w) & ~df_tts["asr_ok"]]["norm"].value_counts().head(8)
        print(f"  Häufigste Whisper-Transkriptionen (ggf. in ASR_EXTRA_ACCEPT aufnehmen): {dict(top)}")
df_tts["auto_keep"] = df_tts["is_target"] & ~df_tts["voice_blocked"] & ~df_tts["low_sim"] & (df_tts["asr_ok"] | df_tts["rescued"])
print("\nAutomatisch behalten pro Zielwort:")
print(df_tts[df_tts["is_target"]].groupby("word")["auto_keep"].agg(["sum", "count"]))

Embeddings:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings:   0%|          | 0/24 [00:00<?, ?it/s]

Embeddings:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings:   0%|          | 0/24 [00:00<?, ?it/s]

Embeddings:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings:   0%|          | 0/24 [00:00<?, ?it/s]

Embeddings:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings:   0%|          | 0/47 [00:00<?, ?it/s]

Echte Clips untereinander (Leave-one-out): Mittel / 5. Perzentil: {'Brav': (0.94, 0.88), 'Hier': (0.96, 0.89), 'Sitz': (0.93, 0.87), 'Such': (0.92, 0.84)}
Brav: Whisper-Quote je Stimme: {'de_DE-eva_k-x_low': 0.04, 'de_DE-karlsson-low': 0.01, 'de_DE-kerstin-low': 0.0, 'de_DE-mls-medium': 0.0, 'de_DE-pavoque-low': 0.62, 'de_DE-ramona-low': 0.0, 'de_DE-thorsten-high': 0.61, 'de_DE-thorsten-medium': 0.96, 'de_DE-thorsten_emotional-medium': 0.02}
  Stimme gesperrt für Brav: de_DE-kerstin-low (Whisper-Quote 0.00, Median 0.02)
  Stimme gesperrt für Brav: de_DE-mls-medium (Whisper-Quote 0.00, Median 0.02)
  Stimme gesperrt für Brav: de_DE-ramona-low (Whisper-Quote 0.00, Median 0.02)
Hier: Whisper-Quote je Stimme: {'de_DE-eva_k-x_low': 0.0, 'de_DE-karlsson-low': 0.26, 'de_DE-kerstin-low': 0.0, 'de_DE-mls-medium': 0.0, 'de_DE-pavoque-low': 0.02, 'de_DE-ramona-low': 0.0, 'de_DE-thorsten-high': 0.78, 'de_DE-thorsten-medium': 0.63, 'de_DE-thorsten_emotional-medium': 0.1}
  Stimme gesperrt für Hier:

## 9. Manuelles Anhören der Grenzfälle
Für jedes trainierte Wort erscheinen drei Gruppen: **A** = Whisper OK, aber geringe Ähnlichkeit (Standard: verwerfen), **B** = Whisper lehnt ab, aber hohe Ähnlichkeit (möglicherweise zu Unrecht abgelehnt, Standard: verwerfen), **C** = Stichprobe der automatisch behaltenen Clips (Standard: behalten). Setze pro Clip „Behalten“ oder „Verwerfen“ und speichere. Entscheidungen bleiben auf Drive erhalten.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
try:
    from google.colab import output as _co
    _co.enable_custom_widget_manager()
except Exception:
    pass

DECISIONS_FILE = CACHE / "review_decisions.json"
DECISIONS = json.loads(DECISIONS_FILE.read_text()) if DECISIONS_FILE.exists() else {}

def review_items(word):
    d = df_tts[df_tts["word"] == word].copy()
    a = d[d["asr_ok"] & ~d["voice_blocked"] & d["low_sim"]].sort_values("sim").head(REVIEW_N_LOW_SIM)
    b = d[~d["asr_ok"] & d["sim"].notna()].sort_values("sim", ascending=False).head(REVIEW_N_RESCUE)
    c = d[d["auto_keep"]].sample(min(REVIEW_N_SPOT, int(d["auto_keep"].sum())), random_state=SEED) if d["auto_keep"].any() else d.head(0)
    items = []
    for tag, df in (("A niedrige Ähnlichkeit", a), ("B von Whisper abgelehnt", b), ("C Stichprobe", c)):
        for r in df.itertuples():
            items.append(dict(key=asr_key(r.path), path=r.path, tag=tag, asr=r.asr, sim=r.sim, auto=bool(r.auto_keep)))
    return items

def review_ui(word):
    items = review_items(word)
    rows, state = [], {}
    for it in items:
        state[it["key"]] = DECISIONS.get(it["key"], it["auto"])
        try:
            player = widgets.Audio.from_file(it["path"])
            player.autoplay, player.loop = False, False
        except Exception:
            player = widgets.HTML("(Audio nicht abspielbar)")
        tb = widgets.ToggleButtons(options=[("Behalten", True), ("Verwerfen", False)], value=state[it["key"]])
        tb.observe(lambda ch, k=it["key"]: state.__setitem__(k, ch["new"]), names="value")
        sim = "–" if pd.isna(it["sim"]) else f"{it['sim']:.2f}"
        info = widgets.HTML(f"<b>{it['tag']}</b> · Whisper: „{it['asr']}“ · Ähnlichkeit {sim}<br><small>{Path(it['path']).name}</small>")
        rows.append(widgets.HBox([player, widgets.VBox([info, tb])]))
    out = widgets.Output()
    btn = widgets.Button(description=f"Entscheidungen für {word} speichern", button_style="success")
    def save(_):
        DECISIONS.update(state)
        DECISIONS_FILE.write_text(json.dumps(DECISIONS, ensure_ascii=False, indent=1))
        with out:
            print(f"{len(state)} Entscheidungen gespeichert ({sum(1 for v in state.values() if not v)} verworfen).")
    btn.on_click(save)
    display(widgets.HTML(f"<h3>{word}: {len(items)} Clips</h3>"), *rows, btn, out)

for w in WORDS_TO_TRAIN:
    review_ui(w)

HTML(value='<h3>Brav: 55 Clips</h3>')

Button(button_style='success', description='Entscheidungen für Brav speichern', style=ButtonStyle())

Output()

## 10. Freigegebene Clips festlegen
Übernimmt die automatische Entscheidung, überschrieben durch deine gespeicherten Entscheidungen. Falls du nichts gespeichert hast, gilt die automatische Auswahl.

In [ ]:
df_tts["keep"] = [DECISIONS.get(asr_key(p), a) for p, a in zip(df_tts["path"], df_tts["auto_keep"])]
ACCEPTED_POS = {w: [Path(p) for p in df_tts[(df_tts["word"] == w) & df_tts["keep"]]["path"]] for w in WORDS}
# Negativ-Wörter: nur Clips, die Whisper nicht als eines der Zielwörter versteht
neg_df = df_tts[~df_tts["is_target"]]
ACCEPTED_NEG = {}
for w in NEG_WORDS:
    ACCEPTED_NEG[w] = [(Path(p), n) for p, n in zip(neg_df[neg_df["word"] == w]["path"], neg_df[neg_df["word"] == w]["norm"])]
(CACHE / "accepted_tts.json").write_text(json.dumps({w: [str(p) for p in ps] for w, ps in ACCEPTED_POS.items()}, indent=1))
print({w: len(v) for w, v in ACCEPTED_POS.items()})
print("Negativ-Clips gesamt:", sum(len(v) for v in ACCEPTED_NEG.values()))

{'Brav': 393, 'Hier': 371, 'Sitz': 433, 'Such': 217}
Negativ-Clips gesamt: 2100


## 11. Deutsche Sprache als Negativdaten
Die ACAV100M-Features sind überwiegend nicht deutsch. Für die Wörter hier ist deutsche Sprache der wichtigste Negativ-Typ. Quellen in dieser Reihenfolge: eigener Ordner `GERMAN_SPEECH_DIR`, dann die Hugging-Face-Datasets aus `GERMAN_HF_SOURCES`. Einige dieser Datasets können je nach Version nicht mehr ladbar sein (ungeprüft, daher try/except). 25 % der Clips werden als Testmaterial für die Fehlaktivierungsrate zurückgehalten und nie trainiert.

In [15]:
def collect_german(hours):
    target = int(hours * 3600 * SR)
    clips, total = [], 0
    if GERMAN_SPEECH_DIR.exists():
        for p in sorted(GERMAN_SPEECH_DIR.rglob("*")):
            if p.suffix.lower() in AUDIO_EXTS and p.is_file():
                try:
                    x = to_int16(load_wav16k(p)); clips.append(x); total += len(x)
                except Exception as e:
                    print("übersprungen:", p.name, repr(e)[:80])
        if clips:
            print(f"Eigene deutsche Sprache: {total / SR / 3600:.2f} h")
    if total < target:
        for name, cfg, split in GERMAN_HF_SOURCES:
            try:
                ds = datasets.load_dataset(name, cfg, split=split, streaming=True, trust_remote_code=True)
                ds = ds.cast_column("audio", datasets.Audio(sampling_rate=16000))
                got = 0
                for row in tqdm(ds, desc=name):
                    x = to_int16(to_mono_16k(row["audio"]["array"], 16000))
                    clips.append(x); total += len(x); got += len(x)
                    if total >= target:
                        break
                if got:
                    print(f"{name}: {got / SR / 3600:.2f} h geladen"); break
            except Exception as e:
                print("Quelle nicht nutzbar:", name, repr(e)[:150])
    return clips

GERMAN_FEATS = FEAT_DIR / "german_speech_features.npy"
GERMAN_HELDOUT = FEAT_DIR / "german_heldout.wav"
USE_GERMAN = False
if GERMAN_FEATS.exists() and GERMAN_HELDOUT.exists():
    USE_GERMAN = True
else:
    clips = collect_german(GERMAN_SPEECH_HOURS)
    if len(clips) >= 20:
        rng = np.random.default_rng(SEED)
        order = rng.permutation(len(clips))
        n_held = max(1, int(0.25 * len(clips)))
        held = [clips[i] for i in order[:n_held]]
        train_clips = [clips[i] for i in order[n_held:]]
        sf.write(str(GERMAN_HELDOUT), np.concatenate(held), SR, subtype="PCM_16")
        stream = np.concatenate(train_clips)
        n_win = len(stream) // WIN
        wins = rng.permutation(n_win)          # gemischte Reihenfolge, train.py liest die Datei sequentiell
        from numpy.lib.format import open_memmap
        fp = open_memmap(str(GERMAN_FEATS), mode="w+", dtype=np.float32, shape=(n_win, 16, 96))
        for i in tqdm(range(0, n_win, 256), desc="Features deutsche Sprache"):
            idx = wins[i:i + 256]
            batch = np.stack([stream[j * WIN:(j + 1) * WIN] for j in idx])
            e = F.embed_clips(batch, batch_size=128, ncpu=2)
            assert e.shape[1:] == (16, 96), f"Unerwartete Feature-Form {e.shape}"
            fp[i:i + len(idx)] = e
            fp.flush()
        del fp
        print(f"Deutsche Sprache: {n_win} Fenster à 2 s Training, {sum(len(h) for h in held) / SR / 3600:.2f} h Test")
        USE_GERMAN = True
    else:
        print("Keine deutsche Sprache gefunden. Lege Dateien in", GERMAN_SPEECH_DIR, "ab oder passe GERMAN_HF_SOURCES an.")
print("USE_GERMAN =", USE_GERMAN)

USE_GERMAN = True


## 12. Trainingsordner bauen und trainieren
Pro Wort entsteht ein Ordner im Format, das `train.py` erwartet (`positive_train`, `positive_test`, `negative_train`, `negative_test`):

- **Positiv:** echte Trainingsclips (mehrfach kopiert, jede Kopie wird anders augmentiert) plus freigegebene TTS-Clips
- **Negativ:** Hard-Negative-Wörter (TTS), die anderen Zielwörter (TTS und echt) und, wenn vorhanden, deutsche Sprache über `feature_data_files`
- Early Stopping nutzt `positive_test`/`negative_test`: Dort liegen echte Clips der Trainingssprecher und TTS, **nie der Validation-Sprecher**

In [16]:
import yaml

def put(src, dst_dir, copies=1):
    for _ in range(copies):
        shutil.copyfile(src, dst_dir / f"{uuid.uuid4().hex}.wav")

def split_put(files, run, prefix, copies, test_copies, test_frac, test_max=None, rng=None):
    files = list(files)
    rng.shuffle(files)
    n_test = int(round(len(files) * test_frac)) if len(files) >= 5 else 0
    if test_max:
        n_test = min(n_test, test_max)
    for p in files[:n_test]:
        put(p, run / f"{prefix}_test", test_copies)
    for p in files[n_test:]:
        put(p, run / f"{prefix}_train", copies)
    return len(files) - n_test, n_test

def build_run_dir(word):
    s = slug(word)
    name = f"{s}{MODEL_SUFFIX}"
    run = RUNS_DIR / name
    if run.exists():
        shutil.rmtree(run)
    for d in ("positive_train", "positive_test", "negative_train", "negative_test"):
        (run / d).mkdir(parents=True)
    rng = random.Random(SEED)
    stats = {}

    real = sorted((REAL_DIR / s / "train").glob("*.wav"))
    if INCLUDE_VALIDATION_SPEAKER:
        print("ACHTUNG: Validation-Sprecher wird mittrainiert, die Auswertung ist dann nicht mehr aussagekräftig.")
        real += sorted((REAL_DIR / s / "val").glob("*.wav"))
    real = [p for p in real if p not in DROPPED_REAL]
    stats["pos_real"] = split_put(real, run, "positive", REAL_DUP, REAL_TEST_DUP, REAL_TEST_FRAC, rng=rng)
    stats["pos_tts"] = split_put(ACCEPTED_POS[word], run, "positive", 1, 1, 0.1, TTS_TEST_MAX, rng)

    neg_tts = [p for w in NEG_WORDS for p, n in ACCEPTED_NEG[w] if n not in ACCEPT[s]]
    stats["neg_words_tts"] = split_put(neg_tts, run, "negative", 1, 1, 0.1, rng=rng)
    other_tts, other_real = [], []
    for ow in WORDS:
        if ow == word:
            continue
        other_tts += ACCEPTED_POS[ow][:N_TTS_AS_NEG]
        other_real += [p for p in sorted((REAL_DIR / slug(ow) / "train").glob("*.wav")) if p not in DROPPED_REAL]
        if INCLUDE_VALIDATION_SPEAKER:
            other_real += sorted((REAL_DIR / slug(ow) / "val").glob("*.wav"))
    stats["neg_other_tts"] = split_put(other_tts, run, "negative", 1, 1, 0.1, rng=rng)
    stats["neg_other_real"] = split_put(other_real, run, "negative", REAL_NEG_DUP, REAL_TEST_DUP, REAL_TEST_FRAC, rng=rng)
    print(name, {k: f"train {a} / test {b}" for k, (a, b) in stats.items()})
    return name, run

def write_config(name, run):
    cfg = yaml.safe_load(open("/content/openwakeword/examples/custom_model.yml"))
    n_pos = len(list((run / "positive_train").glob("*.wav")))
    ffiles = {"ACAV100M_sample": str(prepare_acav_subset())}
    batch = {"ACAV100M_sample": BATCH_N["ACAV100M_sample"], "adversarial_negative": BATCH_N["adversarial_negative"],
             "positive": BATCH_N["positive"]}
    if USE_GERMAN:
        ffiles["german_speech"] = str(GERMAN_FEATS)
        batch["german_speech"] = BATCH_N["german_speech"]
    cfg.update(
        model_name=name, target_phrase=[name.replace(MODEL_SUFFIX, "")], custom_negative_phrases=[],
        n_samples=n_pos, n_samples_val=max(1, len(list((run / "positive_test").glob("*.wav")))),
        piper_sample_generator_path=str(STUBS), output_dir=str(RUNS_DIR),
        rir_paths=[str(RIR_DIR)], background_paths=BACKGROUND_DIRS,
        background_paths_duplication_rate=[1] * len(BACKGROUND_DIRS),
        false_positive_validation_data_path=str(FPVAL), augmentation_rounds=AUG_ROUNDS,
        feature_data_files=ffiles, batch_n_per_class=batch, layer_size=LAYER_SIZE, steps=STEPS,
        max_negative_weight=MAX_NEGATIVE_WEIGHT, target_false_positives_per_hour=TARGET_FP_PER_HOUR)
    path = RUNS_DIR / f"{name}.yaml"
    path.write_text(yaml.safe_dump(cfg, allow_unicode=True))
    return path

RESUME_TRAINING = False   # True: Features des letzten Laufs sind noch da, nur das Training wiederholen (spart ca. 6 Minuten)

def free_memory():
    """Gibt Notebook-Speicher frei, den das Training nicht braucht (Whisper, Piper-Stimmen)."""
    import gc
    for n in ("ASR", "VOICE_POOL", "clips", "held", "train_clips", "stream", "wins", "ds", "batch", "x"):
        if n in globals():
            del globals()[n]
    sys.last_type = sys.last_value = sys.last_traceback = None   # frühere Fehlermeldungen halten sonst Objekte fest
    gc.collect()
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    try:
        import psutil
        m = psutil.virtual_memory()
        print(f"Arbeitsspeicher frei: {m.available / 1e9:.1f} GB von {m.total / 1e9:.1f} GB")
    except Exception:
        pass

FEATURE_FILES = ["positive_features_train.npy", "negative_features_train.npy",
                 "positive_features_test.npy", "negative_features_test.npy"]

def train_word(word, resume=RESUME_TRAINING):
    free_memory()
    name = f"{slug(word)}{MODEL_SUFFIX}"
    run = RUNS_DIR / name
    if resume and all((run / f).exists() for f in FEATURE_FILES) and (RUNS_DIR / f"{name}.yaml").exists():
        print("Fortsetzen: vorhandene Features werden verwendet.")
        cfg = RUNS_DIR / f"{name}.yaml"
        base_cmd = f"{sys.executable} {STUBS / 'run_train.py'} --training_config {cfg}"
    else:
        name, run = build_run_dir(word)
        cfg = write_config(name, run)
        base_cmd = f"{sys.executable} {STUBS / 'run_train.py'} --training_config {cfg}"
        sh(f"{base_cmd} --augment_clips --overwrite", cwd=str(WORK))
    sh(f"{base_cmd} --train_model", cwd=str(WORK))
    onnx = RUNS_DIR / f"{name}.onnx"
    assert onnx.exists(), f"Kein Modell erzeugt: {onnx}"
    shutil.copyfile(onnx, MODELS_OUT / onnx.name)
    print("Modell gespeichert:", MODELS_OUT / onnx.name)
    return name

TRAINED = [train_word(w) for w in WORDS_TO_TRAIN]

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Training:  10%|▉         | 4765/50000 [00:52<08:25, 89.44it/s]
[RAM] frei 43.9 GB | Training 1.8 GB | Worker 19.3 GB

Training:  21%|██        | 10312/50000 [01:53<07:02, 93.92it/s]
[RAM] frei 43.9 GB | Training 1.8 GB | Worker 19.3 GB

Training:  33%|███▎      | 16424/50000 [02:52<05:32, 100.88it/s]
[RAM] frei 43.9 GB | Training 1.8 GB | Worker 19.3 GB

Training:  46%|████▌     | 22964/50000 [03:53<04:09, 108.22it/s]
[RAM] frei 43.9 GB | Training 1.8 GB | Worker 19.3 GB

Training:  59%|█████▉    | 29534/50000 [04:52<02:59, 113.73it/s]
[RAM] frei 43.9 GB | Training 1.8 GB | Worker 19.3 GB

Training:  73%|███████▎  | 36373/50000 [05:52<01:58, 115.38it/s]
[RAM] frei 43.9 GB | Training 1.8 GB | Worker 19.3 GB

Training:  84%|████████▍ | 42101/50000 [06:52<01:08, 115.66it/s]
[RAM] frei 43.8 GB | Training 2.1 GB | Worker 19.3 GB

Training:  95%|█████████▌| 47715/50000 [07:53<00:19, 116.49it/s]
[RAM] frei 43.9 GB | Training 2

## 12b. Training nach einem Neustart der Sitzung (bei Abbruch mit „Killed“ / Exit 137)
Wenn das Training wegen Arbeitsspeicher abbricht, hilft ein **Neustart der Sitzung** (Laufzeit > Sitzung neu starten, nicht „Laufzeit trennen und löschen“). Dateien auf der Platte und auf Drive bleiben erhalten, nur der Speicher des Notebooks wird geleert. Danach in dieser Reihenfolge ausführen: Abschnitt 0 (Setup), 1 (Konfiguration), 2 (Hilfsfunktionen), 4 (Wrapper), diese Zelle, dann Abschnitt 13 (Auswertung). Alles andere wird nicht gebraucht.

In [ ]:
WORD_RESUME = "Sitz"
name = f"{slug(WORD_RESUME)}{MODEL_SUFFIX}"
cfg = RUNS_DIR / f"{name}.yaml"
need = [RUNS_DIR / name / f for f in ("positive_features_train.npy", "negative_features_train.npy",
                                      "positive_features_test.npy", "negative_features_test.npy")]
assert cfg.exists() and all(p.exists() for p in need), \
    "Features oder Konfiguration fehlen (Laufzeit wurde gelöscht?). Dann Abschnitt 12 normal ausführen (RESUME_TRAINING = False)."
# Kleinere ACAV-Stichprobe in die vorhandene Konfiguration eintragen (verhindert, dass der RAM beim Training vollläuft)
import yaml
c = yaml.safe_load(open(cfg))
c["feature_data_files"]["ACAV100M_sample"] = str(prepare_acav_subset())
open(cfg, "w").write(yaml.safe_dump(c, allow_unicode=True))
print("ACAV:", c["feature_data_files"]["ACAV100M_sample"])
try:
    import psutil
    m = psutil.virtual_memory()
    print(f"Arbeitsspeicher frei: {m.available / 1e9:.1f} GB von {m.total / 1e9:.1f} GB")
except Exception:
    pass
sh(f"{sys.executable} {STUBS / 'run_train.py'} --training_config {cfg} --train_model", cwd=str(WORK))
onnx = RUNS_DIR / f"{name}.onnx"
assert onnx.exists(), f"Kein Modell erzeugt: {onnx}"
shutil.copyfile(onnx, MODELS_OUT / onnx.name)
print("Modell gespeichert:", MODELS_OUT / onnx.name)

# Variablen, die Abschnitt 13 nach einem Neustart braucht
from openwakeword.utils import AudioFeatures
F = AudioFeatures(device="cpu")
FPVAL = FEAT_DIR / "validation_set_features.npy"
GERMAN_HELDOUT = FEAT_DIR / "german_heldout.wav"
TRAINED = [name]

AssertionError: Features oder Konfiguration fehlen (Laufzeit wurde gelöscht?). Dann Abschnitt 12 normal ausführen (RESUME_TRAINING = False).

## 12c. Fertiges Modell übernehmen (ohne neues Training)
Wenn das Training bis zum Schluss durchgelaufen ist und nur die tflite-Umwandlung am Ende fehlgeschlagen ist, liegt das `.onnx`-Modell schon auf der Platte. Diese Zelle kopiert es nach Drive und stellt die Variablen für Abschnitt 13 her.

In [ ]:
name = f"{slug(WORDS_TO_TRAIN[0])}{MODEL_SUFFIX}"
onnx = RUNS_DIR / f"{name}.onnx"
assert onnx.exists(), f"{onnx} fehlt: das Training ist nicht bis zum Export gekommen"
shutil.copyfile(onnx, MODELS_OUT / onnx.name)
print("Modell gespeichert:", MODELS_OUT / onnx.name)
from openwakeword.utils import AudioFeatures
if "F" not in globals():
    F = AudioFeatures(device="cpu")
FPVAL = FEAT_DIR / "validation_set_features.npy"
GERMAN_HELDOUT = FEAT_DIR / "german_heldout.wav"
TRAINED = [name]

Modell gespeichert: /content/drive/MyDrive/models/sitz_de.onnx


## 13. Auswertung
Gemessen wird an Material, das nie im Training war:

- **Recall:** Aufnahmen des Validation-Sprechers (gleiches Wort)
- **Cross-Word-Fehlaktivierungen:** Aufnahmen des Validation-Sprechers von den *anderen* Wörtern
- **Fehlaktivierungen pro Stunde** auf zurückgehaltener deutscher Sprache und auf dem openWakeWord-Validierungsset (Englisch, Musik, Geräusche)

Daraus ergibt sich pro Schwelle eine Tabelle. Die empfohlene Schwelle ist die niedrigste, bei der Cross-Word-Fehlaktivierungen 0 sind und die Fehlaktivierungen pro Stunde auf deutscher Sprache unter `EVAL_FP_PER_HOUR` liegen. Die Schwelle ist an einem einzelnen Sprecher bestimmt und deshalb nur ein Richtwert, teste sie im echten Einsatz (Roboter-Mikrofon, Abstand, Motorgeräusche).

In [17]:
import onnxruntime as ort
EVAL_FP_PER_HOUR = 0.5
THRESHOLDS = [round(t, 2) for t in np.arange(0.05, 1.0, 0.05)]

class Head:
    """Das trainierte ONNX-Modell. train.py exportiert ohne dynamische Batch-Achse, dann wird Fenster für Fenster gerechnet."""
    def __init__(self, path):
        self.sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
        self.name = self.sess.get_inputs()[0].name
        try:
            self.sess.run(None, {self.name: np.zeros((2, 16, 96), np.float32)})
            self.batched = True
        except Exception:
            self.batched = False

    def __call__(self, win):
        win = np.ascontiguousarray(win, dtype=np.float32)
        if self.batched:
            return self.sess.run(None, {self.name: win})[0].reshape(-1)
        return np.array([self.sess.run(None, {self.name: win[i:i + 1]})[0].reshape(-1)[0] for i in range(len(win))], np.float32)

def head_scores(head, emb, chunk=4096):
    """emb: (frames, 96) -> Score je 16-Frame-Fenster (Schritt 80 ms)."""
    if len(emb) < 16:
        return np.zeros(0, np.float32)
    win = np.lib.stride_tricks.sliding_window_view(emb.astype(np.float32), (16, 96))[:, 0]
    return np.concatenate([head(win[i:i + chunk]) for i in range(0, len(win), chunk)])

def clip_max_score_batch(sess, x_int16):
    """Eigene Variante mit Batch-Features (nur noch für den Vergleich im sanity_check)."""
    x = np.concatenate([np.zeros(SR, np.int16), x_int16, np.zeros(SR, np.int16)])
    return float(head_scores(sess, F._get_embeddings(x)).max(initial=0.0))

OWW = {}
def get_oww(path):
    if str(path) not in OWW:
        OWW[str(path)] = openwakeword.Model(wakeword_models=[str(path)], inference_framework="onnx")
    return OWW[str(path)]

def stream_scores(oww, x_int16, pad=True):
    """Echte openWakeWord-Pipeline (Streaming, 1280-Sample-Blöcke). Gibt Score je 80-ms-Block zurück."""
    oww.reset()
    preds = oww.predict_clip(x_int16, padding=1 if pad else 0)
    return np.array([list(d.values())[0] for d in preds], np.float32)

def clip_max_score(oww, x_int16):
    return float(stream_scores(oww, x_int16).max(initial=0.0))

def count_events(scores, thr, refractory=25):
    n, last = 0, -10**9
    for i in np.flatnonzero(scores >= thr):
        if i - last >= refractory:
            n, last = n + 1, i
    return n

def embed_long(x_int16, seg=1280 * 250, ctx=1280 * 12):
    """Embeddings für langes Audio in Stücken (ca. 20 s, mit ca. 1 s Überlapp, ohne Lücken und ohne Doppelte).
    F._get_embeddings schickt sonst alle Fenster auf einmal durch das Modell (bei 10 Minuten viele GB RAM)."""
    out, next_k, pos = [], 0, 0
    while pos < len(x_int16):
        base = max(0, pos - ctx)                      # immer ein Vielfaches von 1280 Samples
        part = x_int16[base:pos + seg]
        if len(part) >= SR:                           # unter 1 s ergibt kein Fenster
            e = F._get_embeddings(part)
            k0 = base // 1280                         # globaler Index des ersten Fensters dieses Stücks
            keep = max(0, next_k - k0)
            if keep < len(e):
                out.append(e[keep:])
            next_k = max(next_k, k0 + len(e))
        pos += seg
    return np.vstack(out) if out else np.zeros((0, 96), np.float32)

def long_scores(oww, x_int16, block=SR * 300):
    """Langes Audio durch die echte Streaming-Pipeline, in 5-Minuten-Stücken ohne Reset dazwischen."""
    oww.reset(); out = []
    for i in range(0, len(x_int16), block):
        part = x_int16[i:i + block]
        part = part[:len(part) // 1280 * 1280 + 1280] if len(part) % 1280 else np.concatenate([part, np.zeros(1280, np.int16)])
        out.append(np.array([list(d.values())[0] for d in oww.predict_clip(part, padding=0)], np.float32))
    return np.concatenate(out)

def feature_scores(sess, feats2d, block=40000):
    out = []
    for i in range(0, len(feats2d) - 16, block):
        out.append(head_scores(sess, np.asarray(feats2d[i:i + block + 15], dtype=np.float32)))
    return np.concatenate(out)

held_audio = to_int16(load_wav16k(GERMAN_HELDOUT)) if GERMAN_HELDOUT.exists() else None
if held_audio is not None:
    print(f"Deutsches Testmaterial: {len(held_audio) / SR / 3600:.2f} h")
fp_feats = np.load(FPVAL, mmap_mode="r")

def evaluate(word):
    s = slug(word); name = f"{s}{MODEL_SUFFIX}"
    sess = Head(MODELS_OUT / f"{name}.onnx")      # nur für die Feature-Datei (englische Geräusche)
    oww = get_oww(MODELS_OUT / f"{name}.onnx")
    pos = [clip_max_score(oww, to_int16(load_wav16k(p))) for p in sorted((REAL_DIR / s / "val").glob("*.wav"))]
    cross = []
    for ow in WORDS:
        if ow != word:
            cross += [clip_max_score(oww, to_int16(load_wav16k(p))) for p in sorted((REAL_DIR / slug(ow) / "val").glob("*.wav"))]
    sc_de = long_scores(oww, held_audio) if held_audio is not None else None
    hrs_de = len(sc_de) * 0.08 / 3600 if sc_de is not None else 0
    sc_en = feature_scores(sess, fp_feats)
    hrs_en = len(sc_en) * 0.08 / 3600
    rows = []
    for t in THRESHOLDS:
        rows.append(dict(schwelle=t,
                         recall_val=np.mean(np.array(pos) >= t) if pos else np.nan,
                         cross_word_fa=int(np.sum(np.array(cross) >= t)) if cross else 0,
                         fp_pro_h_deutsch=count_events(sc_de, t) / hrs_de if hrs_de else np.nan,
                         fp_pro_h_en_geraeusche=count_events(sc_en, t) / hrs_en))
    df = pd.DataFrame(rows)
    ok = df[(df["cross_word_fa"] == 0) & ((df["fp_pro_h_deutsch"] <= EVAL_FP_PER_HOUR) | df["fp_pro_h_deutsch"].isna())]
    rec = float(ok["schwelle"].min()) if len(ok) else None
    print(f"\n=== {name}: {len(pos)} Validation-Clips, {len(cross)} Cross-Word-Clips, "
          f"{hrs_de:.2f} h deutsche Sprache, {hrs_en:.1f} h Englisch/Geräusche ===")
    display(df.round(3))
    print("Empfohlene Schwelle:", rec if rec is not None else "keine Schwelle erfüllt alle Kriterien, siehe Tabelle")
    return df, rec

# Kontrolle: stimmt die eigene Auswertung mit der echten openWakeWord-Pipeline überein?
def sanity_check(word):
    """Vergleich: echte Streaming-Pipeline vs. Batch-Features (wie im Training) auf den ersten Validation-Clips."""
    s_ = slug(word); name = f"{s_}{MODEL_SUFFIX}"
    path = MODELS_OUT / f"{name}.onnx"
    oww = get_oww(path); sess = Head(path)
    for p in sorted((REAL_DIR / s_ / "val").glob("*.wav"))[:3]:
        x = to_int16(load_wav16k(p))
        print(f"{p.name}: Streaming (echt) {clip_max_score(oww, x):.3f} | Batch-Features {clip_max_score_batch(sess, x):.3f}")

EVAL = {}
for name, w in zip(TRAINED, WORDS_TO_TRAIN):
    EVAL[w] = evaluate(w)
    sanity_check(w)

Deutsches Testmaterial: 0.94 h

=== brav_de: 11 Validation-Clips, 32 Cross-Word-Clips, 0.94 h deutsche Sprache, 10.7 h Englisch/Geräusche ===


,schwelle,recall_val,cross_word_fa,fp_pro_h_deutsch,fp_pro_h_en_geraeusche
0,0.05,0.727,0,1.06,35.340
1,0.10,0.727,0,1.06,30.291
2,0.15,0.727,0,1.06,26.458
3,0.20,0.727,0,1.06,24.775
4,0.25,0.727,0,1.06,22.812
5,0.30,0.727,0,1.06,22.064
6,0.35,0.727,0,1.06,20.755
7,0.40,0.727,0,1.06,20.101
8,0.45,0.727,0,1.06,19.446
9,0.50,0.727,0,1.06,18.979


Empfohlene Schwelle: 0.85
val_000_Manuel_Brav_01.wav: Streaming (echt) 0.742 | Batch-Features 0.007
val_001_Manuel_Brav_02.wav: Streaming (echt) 0.025 | Batch-Features 0.000
val_002_Manuel_Brav_03.wav: Streaming (echt) 0.001 | Batch-Features 0.005


## 14. Verwendung

Die Modelle liegen in `MyDrive/wakeword_de/models/`. Einbindung wie bisher, mit dem Namen als Schlüssel:

```python
import openwakeword
oww = openwakeword.Model(wakeword_models=["sitz_de.onnx"], inference_framework="onnx")
score = oww.predict(frame_int16_1280)["sitz_de"]   # 1280 Samples = 80 ms bei 16 kHz
if score >= SCHWELLE: ...
```

Nächste Schritte nach dem ersten Wort: `WORDS_TO_TRAIN` erweitern und ab Abschnitt 6 (TTS) erneut ausführen (vorhandene Clips werden wiederverwendet). Wenn die Einstellungen feststehen: `INCLUDE_VALIDATION_SPEAKER = True`, `STEPS = 50000` und neu trainieren. Die Schwellen aus der Auswertung gelten dann als Richtwert.